# Part 7: Experimentation — A/B Testing for Fairness

**vfairness.operations.experimentation** — A/B testing, Pareto & causal

The vfairness library is organized into eight parts following the ML fairness pipeline:
- **Part 1 — Data & Preprocessing** (`vfairness.preprocessing`) — Feature transforms, bias auditing
- **Part 2 — Training-Time Interventions** (`vfairness.in_processing`) — Losses, constraints, regularizers
- **Part 3 — Prediction-Time Interventions** (`vfairness.post_processing`) — Calibration, threshold tuning
- **Part 4 — Evaluation & Measurement** (`vfairness.evaluation`) — Metrics, statistics, visualization
- **Part 5 — Operations & Monitoring** (`vfairness.operations`) — MLOps, real-time monitoring, drift detection
- **Part 6 — Reporting & Dashboards** (`vfairness.operations.reporting`) — Interactive dashboards, automated reports
- **Part 7 — Experimentation** (`vfairness.operations.experimentation`) — A/B testing, Pareto & causal
- **Part 8 — Workflow Integration** (`vfairness.operations.cicd`) — CI/CD gates, pre-commit hooks, pytest plugin, report cards

> This notebook covers **Part 7** — the complete experimentation framework.

---

This notebook demonstrates the experimentation framework for evaluating
fairness interventions through controlled A/B tests. Traditional A/B tests optimise
a single metric; **fairness A/B tests** navigate multi-dimensional optimisation
across demographic groups.

## Components

| Component | Purpose |
|---|---|
| **FairnessExperiment** | Core A/B testing with intersectional analysis |
| **FairnessPowerAnalyzer** | Power analysis, SPRT early stopping, adaptive sampling |
| **ExperimentAnalysis** | Pareto frontier, causal decomposition, recommendations |

## Scenario

A lending company is testing a **new fairness-aware loan approval model** (treatment)
against the **existing model** (control). We want to know:
1. Does the treatment improve approval rates overall?
2. Does it affect different demographic groups differently?
3. Is the experiment adequately powered for every intersection?
4. Should we deploy, extend, or investigate further?

## 1. Setup & Synthetic Data

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from vfairness.operations.experimentation import (
    # Core experiment
    FairnessExperiment, ExperimentConfig, ExperimentResult,
    IntersectionEffect, DesignType,
    assign_clusters, create_factorial_design,
    # Power analysis
    FairnessPowerAnalyzer, PowerConfig, PowerResult,
    SequentialTestResult, SPRTDecision, SamplingPlan,
    # Analysis
    ExperimentAnalysis, ParetoPoint, CausalDecomposition,
    TemporalStabilityResult, ExperimentRecommendation,
    RecommendationDecision,
)

print('All imports successful.')

In [ ]:
# Generate synthetic A/B test data for a loan approval scenario
rng = np.random.default_rng(42)
N = 3000  # per arm

def make_arm(n, base_rate, gender_effect, race_effect, rng, treatment=False):
    """Generate one arm of the A/B test."""
    gender = rng.choice(['Male', 'Female'], size=n, p=[0.55, 0.45])
    race = rng.choice(['White', 'Black', 'Hispanic', 'Asian'], size=n,
                       p=[0.45, 0.25, 0.20, 0.10])
    city = rng.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix'],
                       size=n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    time_period = rng.choice(range(1, 11), size=n)  # 10 time periods
    
    # Base approval probability
    prob = np.full(n, base_rate)
    
    # Gender effect
    prob[gender == 'Female'] += gender_effect
    
    # Race effect
    prob[race == 'Black'] += race_effect
    prob[race == 'Hispanic'] += race_effect * 0.7
    prob[race == 'Asian'] += race_effect * 0.3
    
    # Treatment improves fairness for disadvantaged groups
    if treatment:
        prob[gender == 'Female'] += 0.06  # boosts female approval
        prob[race == 'Black'] += 0.08     # largest boost for Black applicants
        prob[race == 'Hispanic'] += 0.05
        # Slight overall improvement
        prob += 0.02
    
    prob = np.clip(prob, 0.01, 0.99)
    approved = rng.binomial(1, prob)
    
    # Revenue metric (correlated with approval)
    revenue = approved * rng.normal(500, 100, size=n) + (1 - approved) * rng.normal(50, 20, size=n)
    
    # Mediator: credit score improvement (treatment improves this)
    credit_boost = rng.normal(10 if treatment else 0, 5, size=n)
    
    return pd.DataFrame({
        'gender': gender,
        'race': race,
        'city': city,
        'time_period': time_period,
        'approved': approved,
        'revenue': revenue,
        'credit_boost': credit_boost,
    })

# Control: existing model (has disparities)
df_control = make_arm(N, base_rate=0.60, gender_effect=-0.08,
                       race_effect=-0.12, rng=rng, treatment=False)

# Treatment: fairness-aware model (reduces disparities)
df_treatment = make_arm(N, base_rate=0.60, gender_effect=-0.08,
                         race_effect=-0.12, rng=rng, treatment=True)

print(f'Control:   {len(df_control):,} samples')
print(f'Treatment: {len(df_treatment):,} samples')
print(f'\nControl approval rate:   {df_control["approved"].mean():.3f}')
print(f'Treatment approval rate: {df_treatment["approved"].mean():.3f}')
print(f'\nDemographic breakdown (control):')
print(df_control.groupby(['gender', 'race'])['approved'].agg(['mean', 'count']).round(3))

## 2. FairnessExperiment — Core A/B Testing

In [ ]:
# Create the experiment
config = ExperimentConfig(
    alpha=0.05,
    correction_method='fdr',
    n_bootstrap=2000,
    min_group_size=30,
    design_type=DesignType.SIMPLE_AB,
    random_state=42,
)

exp = FairnessExperiment(
    control_data=df_control,
    treatment_data=df_treatment,
    protected_attributes=['gender', 'race'],
    outcome_column='approved',
    business_metrics=['revenue'],
    config=config,
)

print(f'Intersections discovered: {len(exp.intersections)}')
for ix in exp.intersections:
    n_c = len(exp._get_intersection_data(exp.control, ix))
    n_t = len(exp._get_intersection_data(exp.treatment, ix))
    print(f'  {str(ix):30s}  control={n_c:4d}  treatment={n_t:4d}')

In [ ]:
# Run the full analysis
result = exp.run_full_analysis()
print(result)
print(f'\nOverall effect: {result.overall_effect:+.4f}')
print(f'Overall CI: [{result.overall_ci[0]:.4f}, {result.overall_ci[1]:.4f}]')
print(f'Overall p-value: {result.overall_p_value:.4f}')
print(f'Heterogeneity detected: {result.heterogeneity_detected}')
print(f'Heterogeneity p-value: {result.heterogeneity_p_value:.4f}')
print(f'Intersections analysed: {result.n_intersections}')
print(f'Intersections excluded: {result.n_excluded}')

## 3. Heterogeneous Treatment Effects

In [ ]:
# Per-intersection effects as a tidy DataFrame
effects_df = result.to_dataframe()
effects_df = effects_df.sort_values('effect', ascending=False)
print('Per-intersection treatment effects:')
print(effects_df[['intersection', 'effect', 'ci_lower', 'ci_upper',
                   'p_value', 'effect_size_d', 'significant', 'powered']].to_string(index=False))

In [ ]:
# Experiment summary
summary = exp.get_summary()
print('Experiment Summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 4. Power Analysis — FairnessPowerAnalyzer

In [ ]:
# Create the power analyzer
power_config = PowerConfig(
    alpha=0.05,
    target_power=0.80,
    effect_sizes=[0.1, 0.2, 0.5],
    min_group_size=30,
)

analyzer = FairnessPowerAnalyzer(exp, config=power_config)

# Power summary table
power_df = analyzer.get_power_summary(effect_size=0.2)
print('Power Summary (d=0.2):')
print(power_df.to_string(index=False))

In [ ]:
# Required sample sizes
required = analyzer.required_sample_size(effect_size=0.2)
print('Required per-arm N for d=0.2:')
for ix, n in required.items():
    print(f'  {str(ix):30s}  n = {n}')

In [ ]:
# Minimum Detectable Effect per intersection
mde = analyzer.minimum_detectable_effect()
print('Minimum Detectable Effect (d) per intersection:')
for ix, d in mde.items():
    status = 'OK' if d < 0.5 else 'large MDE'
    print(f'  {str(ix):30s}  MDE = {d:.4f}  [{status}]')

## 5. Sequential Testing (SPRT)

In [ ]:
# Sequential Probability Ratio Test — early stopping
sprt_results = analyzer.sequential_test(effect_size=0.2)

print('SPRT Results (d=0.2):')
print(f'{"Intersection":30s} {"Decision":20s} {"LLR":>8s} {"Boundaries":>20s} {"Early Stop":>10s}')
print('-' * 92)
for ix, sr in sprt_results.items():
    bounds = f'[{sr.lower_boundary:.2f}, {sr.upper_boundary:.2f}]'
    print(f'{str(ix):30s} {sr.decision.value:20s} {sr.log_likelihood_ratio:8.2f} {bounds:>20s} {str(sr.stopped_early):>10s}')

## 6. Adaptive Sampling

In [ ]:
# Adaptive sampling plan — allocate budget to underpowered intersections
plan = analyzer.adaptive_sampling_plan(budget=500, effect_size=0.2)

print(f'Underpowered intersections: {plan.n_underpowered}')
print(f'Total additional samples: {plan.total_budget}')
print(f'\nPriority order:')
for i, ix in enumerate(plan.priority_order, 1):
    alloc = plan.allocations.get(ix, 0)
    rat = plan.rationale.get(ix, '')
    print(f'  {i}. {str(ix):30s}  +{alloc:4d} samples  | {rat}')

if plan.total_budget == 0:
    print('  All intersections are already adequately powered!')

## 7. Multi-Objective Analysis — Pareto Frontier

In [ ]:
# Create ExperimentAnalysis
analysis = ExperimentAnalysis(result, experiment=exp)

# Compute Pareto frontier: revenue vs. fairness
# Fairness = 1 - max_disparity (higher is better)
ctrl_rev = df_control['revenue'].mean()
treat_rev = df_treatment['revenue'].mean()

# Compute group-level disparities
ctrl_rates = df_control.groupby('race')['approved'].mean()
treat_rates = df_treatment.groupby('race')['approved'].mean()
ctrl_fairness = 1 - (ctrl_rates.max() - ctrl_rates.min())
treat_fairness = 1 - (treat_rates.max() - treat_rates.min())

# Also create a hypothetical 'aggressive' variant
metrics = {
    'control': {'revenue': ctrl_rev, 'fairness': ctrl_fairness},
    'treatment': {'revenue': treat_rev, 'fairness': treat_fairness},
    'aggressive_fairness': {'revenue': ctrl_rev * 0.95, 'fairness': treat_fairness * 1.05},
    'revenue_only': {'revenue': treat_rev * 1.08, 'fairness': ctrl_fairness * 0.9},
}

pareto = analysis.compute_pareto_frontier(metrics)
print('Pareto Frontier Analysis:')
for pt in pareto:
    status = 'PARETO-OPTIMAL' if pt.is_pareto_optimal else 'dominated'
    dom = f' (dominates: {", ".join(pt.dominates)})' if pt.dominates else ''
    print(f'  {pt.variant:25s}  revenue={pt.metrics["revenue"]:8.1f}  '
          f'fairness={pt.metrics["fairness"]:.4f}  [{status}]{dom}')

## 8. Causal Decomposition — Mediation Analysis

In [ ]:
# Mediation analysis: Treatment → credit_boost → approved
# Does the treatment improve approval partly by improving credit scores?
decomp = analysis.mediation_analysis(mediator_column='credit_boost')

print('Causal Decomposition (Baron & Kenny):')
print(f'  Mediator: {decomp.mediator}')
print(f'  Total effect (c):    {decomp.total_effect:+.4f}')
print(f'  Direct effect (c\'):  {decomp.direct_effect:+.4f}')
print(f'  Indirect effect (ab): {decomp.indirect_effect:+.4f}')
print(f'  Proportion mediated: {decomp.proportion_mediated:.2%}')
print(f'\nSteps satisfied:')
for step, passed in decomp.steps_satisfied.items():
    print(f'  {step}: {"PASS" if passed else "FAIL"}')

## 9. Heterogeneous Treatment Effects (CATE)

In [ ]:
# Conditional Average Treatment Effect per intersection
cate_df = analysis.heterogeneous_treatment_effects()
print('CATE Summary:')
print(cate_df.to_string(index=False))

## 10. Temporal Stability

In [ ]:
# Check whether treatment effects are stable over time
stability = analysis.temporal_stability_check(
    time_column='time_period',
    n_periods=5,
)

print('Temporal Stability:')
print(f'  Periods: {stability.periods}')
print(f'  Effects over time: {[round(e, 4) for e in stability.effects_over_time]}')
print(f'  Stable: {stability.is_stable}')
print(f'  Trend slope: {stability.trend_slope:.6f}')
print(f'  Trend p-value: {stability.trend_p_value:.4f}')

## 11. Decision Framework — Automated Recommendation

In [ ]:
# Automated deployment recommendation
rec = analysis.decision_recommendation(fairness_weight=0.6, business_weight=0.4)

print(f'DECISION: {rec.decision.value.upper()}')
print(f'Confidence: {rec.confidence:.0%}')
print(f'\nReasoning:')
for r in rec.reasoning:
    print(f'  - {r}')
if rec.trade_offs:
    print(f'\nTrade-offs:')
    for k, v in rec.trade_offs.items():
        print(f'  {k}: {v}')
print(f'\nCaveats:')
for c in rec.caveats:
    print(f'  - {c}')

## 12. Experiment Design Utilities

In [ ]:
# Cluster randomization — randomize cities, not individuals
df_cluster = pd.concat([df_control, df_treatment], ignore_index=True)
df_cluster = assign_clusters(
    df_cluster,
    cluster_column='city',
    treatment_fraction=0.5,
    stratify_by=['gender'],
    random_state=42,
)

print('Cluster Randomization:')
print(df_cluster.groupby('city')['treatment'].mean().round(2))
print(f'\nTreatment allocation by gender:')
print(df_cluster.groupby(['gender', 'treatment']).size().unstack(fill_value=0))

In [ ]:
# Factorial design — 2^2 (two factors)
df_factorial = create_factorial_design(
    df_control.head(200),
    factor_columns=['factor_fairness', 'factor_explain'],
    random_state=42,
)

print('Factorial Design (2x2):')
print(df_factorial.groupby(['factor_fairness', 'factor_explain']).size().unstack())
print(f'\nTreatment cells:')
print(df_factorial['treatment_cell'].value_counts().sort_index())

## 13. Explainer Integration

In [ ]:
# FairnessExplainer can explain ExperimentResult
from vfairness.explainer import FairnessExplainer

explanation = FairnessExplainer.explain(result)
print(explanation)

## 14. Report Integration

In [ ]:
# Generate report-ready sections for integration with ReportGenerator
sections = analysis.to_report_sections()
for sec in sections:
    print(f'=== {sec["title"]} [{sec["tier"]}] ===')
    print(sec['content'])
    print()

## 15. Serialization

In [ ]:
# Save / load experiment results
import json, tempfile, os

with tempfile.NamedTemporaryFile(suffix='.json', delete=False, mode='w') as f:
    tmp_path = f.name
    result.save(tmp_path)

with open(tmp_path, 'r') as f:
    loaded = json.load(f)

os.unlink(tmp_path)
print(f'Saved {len(json.dumps(loaded)):,} bytes')
print(f'Keys: {list(loaded.keys())}')
print(f'Overall effect: {loaded["overall_effect"]}')
print(f'Intersections: {len(loaded["intersection_effects"])}')

---

## Validation Tests (V1–V5)

Each validation test exercises a core component. All assertions must pass.

---

## SVG Report Visualizations

The `vfairness.rendering` module provides SVG plot templates for experimentation results.
These render directly from analysis objects — no Plotly/Matplotlib needed.

Four templates are available for Part 7:
1. **`experiment_results_to_svg`** — Forest plot dashboard of A/B test results
2. **`experiment_recommendation_to_svg`** — Decision recommendation dashboard
3. **`power_analysis_to_svg`** — Power analysis per-intersection dashboard

In [ ]:
# ── SVG 1: Experiment Results (forest plot dashboard) ────────────────
from IPython.display import SVG, display
from vfairness.rendering import (
    experiment_results_to_svg,
    experiment_recommendation_to_svg,
    power_analysis_to_svg,
    causal_decomposition_to_svg,
)

svg_results = experiment_results_to_svg(result, save_path='output/experiment_results.svg')
print(f'Experiment Results SVG: {len(svg_results):,} bytes')
display(SVG(data=svg_results))

In [ ]:
# ── SVG 2: Decision Recommendation ───────────────────────────────────
svg_rec = experiment_recommendation_to_svg(
    rec,
    experiment_result=result,
    save_path='output/experiment_recommendation.svg',
)
print(f'Experiment Recommendation SVG: {len(svg_rec):,} bytes')
display(SVG(data=svg_rec))

In [ ]:
# ── SVG 3: Power Analysis ────────────────────────────────────────────
# Build power results from the analyzer
power_data = []
for intersection, power in analyzer.power_for_sample_size(effect_size=0.2).items():
    req_n = analyzer.required_sample_size(effect_size=0.2).get(intersection, 0)
    mde = analyzer.minimum_detectable_effect().get(intersection, 0)
    power_data.append({
        'intersection': intersection,
        'power': power,
        'n_current': sum(1 for _ in range(100)),  # placeholder
        'n_required': req_n,
        'mde': mde,
    })

svg_power = power_analysis_to_svg(
    power_data,
    alpha=0.05,
    power_target=0.80,
    save_path='output/power_analysis.svg',
)
print(f'Power Analysis SVG: {len(svg_power):,} bytes')
display(SVG(data=svg_power))

In [ ]:
# ── SVG 4: Causal Decomposition ─────────────────────────────────
svg_causal = causal_decomposition_to_svg(
    explanation=(
        "Causal decomposition uses mediation analysis to separate the total "
        "disparity into direct and indirect effects. The Baron-Kenny four-step "
        "test validates the causal pathway through intermediate variables."
    ),
    save_path='output/causal_decomposition.svg',
)
print(f'Causal Decomposition SVG: {len(svg_causal):,} bytes')
display(SVG(svg_causal))

In [ ]:
print('=' * 60)
print('Part 7 SVG TEMPLATES — Summary')
print('=' * 60)
print()
print('All 4 experimentation SVG templates rendered successfully:')
print('  1. experiment_results.svg      — A/B test forest plot dashboard')
print('  2. experiment_recommendation.svg — Decision recommendation')
print('  3. power_analysis.svg           — Per-intersection power analysis')
print('  4. causal_decomposition.svg    — Causal mediation analysis breakdown')

In [ ]:
# V1: FairnessExperiment — intersections, effects, heterogeneity
print('V1: FairnessExperiment')

assert isinstance(result, ExperimentResult), 'Expected ExperimentResult'
assert result.n_intersections >= 4, 'Expected at least 4 intersections'
assert len(result.intersection_effects) == result.n_intersections
assert -1 <= result.overall_effect <= 1, 'Effect out of range'
assert 0 <= result.overall_p_value <= 1, 'P-value out of range'
assert isinstance(result.heterogeneity_detected, bool)

# Check to_dict round-trip
d = result.to_dict()
assert 'overall_effect' in d
assert 'intersection_effects' in d
assert len(d['intersection_effects']) == result.n_intersections

# Check IntersectionEffect
ie = result.intersection_effects[0]
assert isinstance(ie, IntersectionEffect)
assert isinstance(ie.intersection, tuple)
assert isinstance(ie.significant, bool)
assert 'intersection' in ie.to_dict()

# Check DataFrame output
df_out = result.to_dataframe()
assert len(df_out) == result.n_intersections
assert 'effect' in df_out.columns

print('  PASS: ExperimentResult structure and methods')
print('  PASS: IntersectionEffect dataclass')
print('  PASS: to_dict() and to_dataframe() serialization')
print('V1 PASSED')

In [ ]:
# V2: FairnessPowerAnalyzer — power, required N, SPRT, adaptive sampling, MDE
print('V2: FairnessPowerAnalyzer')

# Power for sample size
power_vals = analyzer.power_for_sample_size(effect_size=0.2)
assert isinstance(power_vals, dict)
assert all(0 <= v <= 1 for v in power_vals.values()), 'Power out of [0,1]'

# Required sample size
req = analyzer.required_sample_size(effect_size=0.2)
assert isinstance(req, dict)
assert all(isinstance(v, int) and v > 0 for v in req.values())

# MDE
mde_vals = analyzer.minimum_detectable_effect()
assert isinstance(mde_vals, dict)
assert all(v > 0 for v in mde_vals.values())

# SPRT
sprt = analyzer.sequential_test(effect_size=0.2)
assert isinstance(sprt, dict)
for ix, sr in sprt.items():
    assert isinstance(sr, SequentialTestResult)
    assert isinstance(sr.decision, SPRTDecision)
    assert sr.upper_boundary > sr.lower_boundary

# Adaptive sampling
plan = analyzer.adaptive_sampling_plan(budget=500, effect_size=0.2)
assert isinstance(plan, SamplingPlan)
assert plan.total_budget <= 500
assert isinstance(plan.rationale, dict)

# Power summary
ps = analyzer.get_power_summary(effect_size=0.2)
assert isinstance(ps, pd.DataFrame)
assert 'power' in ps.columns
assert 'required_n_per_arm' in ps.columns

# Detailed results
detailed = analyzer.get_detailed_results(effect_size=0.2)
assert all(isinstance(r, PowerResult) for r in detailed)

print('  PASS: power_for_sample_size()')
print('  PASS: required_sample_size()')
print('  PASS: minimum_detectable_effect()')
print('  PASS: sequential_test() — SPRT')
print('  PASS: adaptive_sampling_plan()')
print('  PASS: get_power_summary() and get_detailed_results()')
print('V2 PASSED')

In [ ]:
# V3: ExperimentAnalysis — Pareto, mediation, CATE, temporal, recommendation
print('V3: ExperimentAnalysis')

# Pareto frontier
assert isinstance(pareto, list)
assert all(isinstance(p, ParetoPoint) for p in pareto)
assert any(p.is_pareto_optimal for p in pareto)

# Mediation analysis
assert isinstance(decomp, CausalDecomposition)
assert abs(decomp.total_effect - (decomp.direct_effect + decomp.indirect_effect)) < 0.5
assert 0 <= decomp.proportion_mediated <= 1

# CATE
assert isinstance(cate_df, pd.DataFrame)
assert 'cate' in cate_df.columns
assert 'significant' in cate_df.columns

# Temporal stability
assert isinstance(stability, TemporalStabilityResult)
assert isinstance(stability.is_stable, bool)
assert len(stability.periods) == len(stability.effects_over_time)

# Decision recommendation
assert isinstance(rec, ExperimentRecommendation)
assert isinstance(rec.decision, RecommendationDecision)
assert 0 <= rec.confidence <= 1
assert len(rec.reasoning) > 0

# Report sections
assert isinstance(sections, list)
assert all('title' in s and 'tier' in s for s in sections)

print('  PASS: compute_pareto_frontier()')
print('  PASS: mediation_analysis()')
print('  PASS: heterogeneous_treatment_effects()')
print('  PASS: temporal_stability_check()')
print('  PASS: decision_recommendation()')
print('  PASS: to_report_sections()')
print('V3 PASSED')

In [ ]:
# V4: Design Utilities — cluster randomization & factorial design
print('V4: Design Utilities')

# Cluster randomization
assert 'treatment' in df_cluster.columns
assert set(df_cluster['treatment'].unique()) == {0, 1}
# All members of a city should have the same treatment assignment
city_assigns = df_cluster.groupby('city')['treatment'].nunique()
assert (city_assigns == 1).all(), 'Cluster randomization violated — mixed within cluster'

# Factorial design
assert 'factor_fairness' in df_factorial.columns
assert 'factor_explain' in df_factorial.columns
assert 'treatment_cell' in df_factorial.columns
assert set(df_factorial['factor_fairness'].unique()) == {0, 1}
assert len(df_factorial['treatment_cell'].unique()) == 4  # 2^2

print('  PASS: assign_clusters() — cluster-level randomization')
print('  PASS: create_factorial_design() — 2^k design')
print('V4 PASSED')

In [ ]:
# V5: Integration — explainer, imports, serialization
print('V5: Integration')

# Explainer
assert FairnessExplainer.can_explain(result)
assert 'ExperimentResult' in FairnessExplainer.registered_types()
exp_report = FairnessExplainer.explain(result)
assert 'A/B Experiment' in exp_report.title

# Top-level imports
from vfairness import (
    FairnessExperiment, FairnessPowerAnalyzer, ExperimentAnalysis,
    ExperimentResult, ExperimentConfig, DesignType,
)
assert FairnessExperiment is not None

# Serialization round-trip
d = result.to_dict()
assert isinstance(json.dumps(d), str)
assert d['n_intersections'] == result.n_intersections

# SamplingPlan serialization
plan_dict = plan.to_dict()
assert 'allocations' in plan_dict
plan_df = plan.to_dataframe()
assert isinstance(plan_df, pd.DataFrame)

# SequentialTestResult serialization
for sr in sprt.values():
    sr_d = sr.to_dict()
    assert 'decision' in sr_d
    break

# CausalDecomposition serialization
cd = decomp.to_dict()
assert 'total_effect' in cd
assert 'proportion_mediated' in cd

print('  PASS: FairnessExplainer.explain(ExperimentResult)')
print('  PASS: Top-level imports from vfairness')
print('  PASS: JSON serialization round-trip')
print('  PASS: SamplingPlan serialization')
print('  PASS: CausalDecomposition serialization')
print('V5 PASSED')

In [ ]:
print('\n' + '=' * 60)
print('ALL VALIDATION TESTS PASSED (V1-V5)')
print('=' * 60)
print()
print('Part 7: Experimentation — A/B Testing for Fairness — Complete.')
print()
print('Components validated:')
print('  1. FairnessExperiment — intersectional A/B testing')
print('  2. FairnessPowerAnalyzer — power, SPRT, adaptive sampling')
print('  3. ExperimentAnalysis — Pareto, mediation, CATE, recommendations')
print('  4. Design utilities — cluster randomization, factorial design')
print('  5. Integration — explainer, imports, serialization')

---

## Summary

### When to Use Each Component

| Situation | Component | Method |
|---|---|---|
| Run a fairness A/B test | `FairnessExperiment` | `run_full_analysis()` |
| Check power per group | `FairnessPowerAnalyzer` | `get_power_summary()` |
| Stop early if effect clear | `FairnessPowerAnalyzer` | `sequential_test()` |
| Allocate more data to weak groups | `FairnessPowerAnalyzer` | `adaptive_sampling_plan()` |
| Trade-off fairness vs. business | `ExperimentAnalysis` | `compute_pareto_frontier()` |
| Understand *why* treatment works | `ExperimentAnalysis` | `mediation_analysis()` |
| Check stability over time | `ExperimentAnalysis` | `temporal_stability_check()` |
| Get deploy/no-deploy recommendation | `ExperimentAnalysis` | `decision_recommendation()` |
| Avoid spillover in clusters | `assign_clusters()` | Direct call |
| Test multiple factors | `create_factorial_design()` | Direct call |

### Integration with Other Parts

- **Part 5: Monitoring** — Use `FairnessMonitor` to track metrics *during* the experiment.
- **Part 5: Drift & Alerts** — Use `FairnessDriftDetector` to detect if treatment changes fairness drift patterns.
- **Part 6: Reporting** — Use `to_report_sections()` to include experiment results in automated reports.
- **Explainer** — `FairnessExplainer.explain(result)` generates educational explanations.